In [3]:
# ==========================================================
# STEP 1 : Import Required Libraries
# Project : Tuberculosis Detection using CoAtNet
# Purpose :
# Import libraries required for data augmentation.
# ==========================================================

import os
import pandas as pd

import torch
from torchvision import transforms

print("Libraries Imported Successfully!")

Libraries Imported Successfully!


In [4]:
# ==========================================================
# STEP 2 : Load Training Metadata
# ==========================================================

train_metadata = pd.read_csv("train_metadata.csv")

print("="*50)
print("Training Metadata Loaded Successfully!")
print("="*50)

print("Training Images :", len(train_metadata))

Training Metadata Loaded Successfully!
Training Images : 560


In [5]:
# ==========================================================
# STEP 3 : Display First Five Records
# ==========================================================

train_metadata.head()

,study_id,gender,age,findings,dataset,label
0,CHNCXR_0185_0.png,Male,44,normal,Shenzhen,0
1,CHNCXR_0243_0.png,Male,27,normal,Shenzhen,0
2,CHNCXR_0627_1.png,Female,26,secondary PTB in the right upper field,Shenzhen,1
3,CHNCXR_0393_1.png,Female,24,"right upper PTB, pleural adhesions in left lo...",Shenzhen,1
4,CHNCXR_0225_0.png,Male,30,normal,Shenzhen,0


In [6]:
# ==========================================================
# STEP 4 : Define Segmented Image Folder
# ==========================================================

image_folder = "Segmented_Images"

print("Image Folder :", image_folder)

print("Folder Exists :", os.path.exists(image_folder))

Image Folder : Segmented_Images
Folder Exists : True


In [12]:
# ==========================================================
# STEP 5 : Data Augmentation Pipeline
# ==========================================================

augmentation = transforms.Compose([

    # Small rotation (±10°)
    transforms.RandomRotation(10),

    # Small translation
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05,0.05)
    ),

    # Slight zoom
    transforms.RandomResizedCrop(
        size=224,
        scale=(0.95,1.05)
    ),

    # Brightness / Contrast
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10
    ),

    # Convert back to Tensor
    transforms.ToTensor()

])

print("Data Augmentation Pipeline Created Successfully!")

Data Augmentation Pipeline Created Successfully!


In [8]:
# ==========================================================
# STEP 6 : Import Required Libraries
# Project : Tuberculosis Detection using CoAtNet
# Purpose :
# Import libraries required for creating a custom PyTorch Dataset.
# ==========================================================

from PIL import Image

from torch.utils.data import Dataset

print("Dataset Libraries Imported Successfully!")

Dataset Libraries Imported Successfully!


In [9]:
# ==========================================================
# STEP 7 : Create Custom Dataset Class
# Project : Tuberculosis Detection using CoAtNet
# Purpose :
# Read segmented chest X-ray images and their labels.
# ==========================================================

class TuberculosisDataset(Dataset):

    def __init__(self, dataframe, image_folder, transform=None):

        self.dataframe = dataframe
        self.image_folder = image_folder
        self.transform = transform

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, index):

        # Get image filename
        image_name = self.dataframe.iloc[index]["study_id"]

        # Create complete image path
        image_path = os.path.join(self.image_folder, image_name)

        # Read image
        image = Image.open(image_path).convert("RGB")

        # Get label
        label = self.dataframe.iloc[index]["label"]

        # Apply augmentation (only for training)
        if self.transform:
            image = self.transform(image)

        return image, label

In [13]:
# ==========================================================
# STEP 8 : Create Training Dataset
# ==========================================================

train_dataset = TuberculosisDataset(

    dataframe=train_metadata,

    image_folder=image_folder,

    transform=augmentation

)

print("=" * 50)
print("Training Dataset Created Successfully!")
print("=" * 50)

print("Total Training Images :", len(train_dataset))

Training Dataset Created Successfully!
Total Training Images : 560


In [14]:
# ==========================================================
# STEP 9 : Verify Dataset
# ==========================================================

image, label = train_dataset[0]

print("Image Shape :", image.shape)
print("Image Data Type :", image.dtype)
print("Label :", label)

Image Shape : torch.Size([3, 224, 224])
Image Data Type : torch.float32
Label : 0


In [15]:
# ==========================================================
# STEP 10 : Import DataLoader
# Project : Tuberculosis Detection using CoAtNet
# Purpose :
# Import DataLoader for batch-wise training.
# ==========================================================

from torch.utils.data import DataLoader

print("DataLoader Imported Successfully!")

DataLoader Imported Successfully!


In [16]:
# ==========================================================
# STEP 11 : Create Training DataLoader
# Purpose :
# Divide training dataset into mini-batches.
# ==========================================================

train_loader = DataLoader(

    dataset=train_dataset,

    batch_size=32,

    shuffle=True,

    num_workers=0

)

print("=" * 50)
print("Training DataLoader Created Successfully!")
print("=" * 50)

print("Batch Size :", train_loader.batch_size)

Training DataLoader Created Successfully!
Batch Size : 32


In [17]:
# ==========================================================
# STEP 12 : Verify Training DataLoader
# ==========================================================

images, labels = next(iter(train_loader))

print("Batch Image Shape :", images.shape)
print("Batch Label Shape :", labels.shape)

Batch Image Shape : torch.Size([32, 3, 224, 224])
Batch Label Shape : torch.Size([32])


In [1]:
# ==========================================================
# STEP 13 : Check TIMM Installation
# ==========================================================

import timm

print("TIMM Version :", timm.__version__)

print("TIMM Imported Successfully!")

C:\Users\Gobika\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TIMM Version : 1.0.28
TIMM Imported Successfully!


In [2]:
# ==========================================================
# STEP 14 : Display Available CoAtNet Models
# Project : Tuberculosis Detection using CoAtNet
# Purpose :
# Display all CoAtNet models available in TIMM.
# ==========================================================

import timm

coatnet_models = timm.list_models("*coat*")

print("=" * 60)
print("Available CoAtNet Models")
print("=" * 60)

for model in coatnet_models:
    print(model)

Available CoAtNet Models
coat_lite_medium
coat_lite_medium_384
coat_lite_mini
coat_lite_small
coat_lite_tiny
coat_mini
coat_small
coat_tiny
coatnet_0_224
coatnet_0_rw_224
coatnet_1_224
coatnet_1_rw_224
coatnet_2_224
coatnet_2_rw_224
coatnet_3_224
coatnet_3_rw_224
coatnet_4_224
coatnet_5_224
coatnet_bn_0_rw_224
coatnet_nano_cc_224
coatnet_nano_rw_224
coatnet_pico_rw_224
coatnet_rmlp_0_rw_224
coatnet_rmlp_1_rw2_224
coatnet_rmlp_1_rw_224
coatnet_rmlp_2_rw_224
coatnet_rmlp_2_rw_384
coatnet_rmlp_3_rw_224
coatnet_rmlp_nano_rw_224
coatnext_nano_rw_224


In [3]:
# ==========================================================
# STEP 15 : Load Pre-trained CoAtNet Model
# Project : Tuberculosis Detection using CoAtNet
# Purpose :
# Load a pre-trained CoAtNet model from TIMM.
# ==========================================================

import timm

# Load Pre-trained CoAtNet
model = timm.create_model(

    "coatnet_bn_0_rw_224",

    pretrained=True

)

print("=" * 60)
print("Pre-trained CoAtNet Loaded Successfully!")
print("=" * 60)

Pre-trained CoAtNet Loaded Successfully!


C:\Users\Gobika\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Gobika\.cache\huggingface\hub\models--timm--coatnet_bn_0_rw_224.sw_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [4]:
# ==========================================================
# STEP 16 : Display CoAtNet Architecture
# Project : Tuberculosis Detection using CoAtNet
# Purpose :
# Display the complete architecture of the pre-trained model.
# ==========================================================

print(model)

MaxxVit(
  (stem): Stem(
    (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (norm1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  )
  (stages): Sequential(
    (0): MaxxVitStage(
      (blocks): Sequential(
        (0): MbConvBlock(
          (shortcut): Downsample2d(
            (pool): AvgPool2d(kernel_size=2, stride=2, padding=0)
            (expand): Conv2d(64, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          )
          (pre_norm): BatchNormAct2d(
            64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (down): Identity()
          (conv1_1x1): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bia

In [5]:
# ==========================================================
# STEP 17 : Modify Final Classification Layer
# Project : Tuberculosis Detection using CoAtNet
# Purpose :
# Replace the original ImageNet classifier (1000 classes)
# with a new classifier for 2 classes:
# 0 - Normal
# 1 - Tuberculosis
# ==========================================================

import torch.nn as nn

# Replace the final fully connected layer

model.head.fc = nn.Linear(

    in_features=768,

    out_features=2

)

print("=" * 60)
print("Final Classification Layer Modified Successfully!")
print("=" * 60)

print(model.head)

Final Classification Layer Modified Successfully!
ClassifierHead(
  (global_pool): SelectAdaptivePool2d(pool_type=avg, flatten=Flatten(start_dim=1, end_dim=-1))
  (drop): Dropout(p=0.0, inplace=False)
  (fc): Linear(in_features=768, out_features=2, bias=True)
  (flatten): Identity()
)


In [6]:
# ==========================================================
# STEP 18 : Detect Device
# Project : Tuberculosis Detection using CoAtNet
# Purpose :
# Detect whether GPU is available.
# ==========================================================

import torch

device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

print("=" * 60)
print("Selected Device :", device)
print("=" * 60)

Selected Device : cpu


In [7]:
# ==========================================================
# STEP 19 : Move Model to Device
# ==========================================================

model = model.to(device)

print("=" * 60)
print("CoAtNet Model Moved to Device Successfully!")
print("=" * 60)

CoAtNet Model Moved to Device Successfully!


In [8]:
# ==========================================================
# STEP 20 : Define Loss Function
# ==========================================================

import torch.nn as nn

criterion = nn.CrossEntropyLoss()

print("=" * 60)
print("Loss Function Created Successfully!")
print("=" * 60)

print(criterion)

Loss Function Created Successfully!
CrossEntropyLoss()


In [9]:
# ==========================================================
# STEP 21 : Define Optimizer
# ==========================================================

import torch.optim as optim

optimizer = optim.Adam(

    model.parameters(),

    lr=0.0001

)

print("=" * 60)
print("Optimizer Created Successfully!")
print("=" * 60)

print(optimizer)

Optimizer Created Successfully!
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)


In [10]:
# ==========================================================
# STEP 22 : Create Validation Dataset
# Project : Tuberculosis Detection using CoAtNet
# Purpose :
# Create validation dataset without augmentation.
# ==========================================================

# Validation dataset

validation_dataset = TuberculosisDataset(

    dataframe=validation_data,

    image_folder=image_folder,

    transform=transforms.Compose([

        transforms.ToTensor()

    ])

)

print("=" * 60)
print("Validation Dataset Created Successfully!")
print("=" * 60)

print("Validation Images :", len(validation_dataset))

NameError: name 'TuberculosisDataset' is not defined